# 07. Feature Engineering, Scalers & Tabular Preprocessing

### Historical Lineage & Theoretical Foundations
- **Lineage:** *Box & Cox (1964); Yeo & Johnson (2000); Micci-Barreca (2001, Target Encoding)*.
- **Core Purpose:** Invertible feature standardization, monotonic power transformations (Yeo-Johnson) for skewed tabular distributions, and serving-time weight folding (graph fusion) to eliminate DRAM roundtrips.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT fit scalers on full unpartitioned datasets:** Fitting on test/validation data causes severe target leakage and optimistic evaluation bias; always `fit` on train and `transform` on test.
> - **DO NOT use standard scaling on heavy-tailed distributions:** Outliers skew $\mu$ and $\sigma$, squashing standard samples into near-zero intervals; use `RobustScaler` or Yeo-Johnson power transforms.
> - **DO NOT run sequential preprocessing passes in production:** Separate scaler kernels incur avoidable DRAM latency; fold linear scaling directly into serving layer weights.


### Mathematical Formulation & Asymptotic Complexity

#### 1. Invertible Feature Standardization
$$\mathbf{z} = \frac{\mathbf{x} - \boldsymbol{\mu}}{\boldsymbol{\sigma} + \epsilon}, \quad \mathbf{x} = \mathbf{z} \odot \boldsymbol{\sigma} + \boldsymbol{\mu}$$

#### 2. Yeo-Johnson Monotonic Power Transform
$$\psi(\lambda, x) = \begin{cases} \frac{(x + 1)^\lambda - 1}{\lambda} & x \ge 0, \lambda \ne 0 \\ \ln(x + 1) & x \ge 0, \lambda = 0 \\ -\frac{(-x + 1)^{2 - \lambda} - 1}{2 - \lambda} & x < 0, \lambda \ne 2 \\ -\ln(-x + 1) & x < 0, \lambda = 2 \end{cases}$$

#### 3. Serving Weight Folding Identity
$$\mathbf{y} = \left( (\mathbf{x} - \boldsymbol{\mu}) \oslash \boldsymbol{\sigma} \right) \mathbf{W}_{\text{head}} + \mathbf{b}_{\text{head}} = \mathbf{x} \mathbf{W}_{\text{fused}} + \mathbf{b}_{\text{fused}}$$
$$\mathbf{W}_{\text{fused}} = \operatorname{diag}(\boldsymbol{\sigma}^{-1}) \mathbf{W}_{\text{head}}, \quad \mathbf{b}_{\text{fused}} = \mathbf{b}_{\text{head}} - (\boldsymbol{\mu} \oslash \boldsymbol{\sigma})^\top \mathbf{W}_{\text{head}}$$

**Asymptotic Complexity:**
- Fitting Pass: Time $\mathcal{O}(N \cdot d)$ summary statistics; Space $\mathcal{O}(d)$.
- Standard Transform: Time $\mathcal{O}(B \cdot d)$; Weight Folding: Time $\mathcal{O}(d \cdot d_{\text{out}})$ one-time precomputation.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

import core

env = core.setup_env(seed=42)
ckpt_path = Path("weights/00_foundations/07_feature_engineering_scalers_and_tabular_preprocessing.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/00_foundations/07_feature_engineering_scalers_and_tabular_preprocessing.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)
dim = int(meta.get("dim", 4))
print(f"Device: {env.device} | Dim: {dim} | Checkpoint: {ckpt_path.name}")


In [ ]:
class StandardScaler(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.register_buffer("mean", torch.zeros(dim), persistent=True)
        self.register_buffer("std", torch.ones(dim), persistent=True)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mean) / self.std

    def inverse_transform(self, z: torch.Tensor) -> torch.Tensor:
        return z * self.std + self.mean

class FusedTabularLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        self.bias = nn.Parameter(torch.empty(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

scaler = StandardScaler(dim=dim)
scaler.mean.copy_(weights["mean"])
scaler.std.copy_(weights["std"])
print("StandardScaler initialized with reference parameters.")


In [ ]:
# 1. Invertible Reconstruction Invariant
z = scaler.transform(test_in)
x_rec = scaler.inverse_transform(z)
torch.testing.assert_close(x_rec, test_in, atol=1e-6, rtol=1e-6)

# 2. Serving Weight Folding Mathematical Identity
w_fused = (1.0 / weights["std"]).unsqueeze(1) * weights["w_head"]
b_fused = weights["b_head"] - (weights["mean"] / weights["std"]) @ weights["w_head"]
fused_model = FusedTabularLinear(dim, 2)
with torch.no_grad():
    fused_model.weight.copy_(w_fused)
    fused_model.bias.copy_(b_fused)

y_explicit = scaler.transform(test_in) @ weights["w_head"] + weights["b_head"]
y_fused = fused_model(test_in)
torch.testing.assert_close(y_fused, y_explicit, atol=1e-5, rtol=1e-5)
torch.testing.assert_close(y_fused, golden_out, atol=1e-5, rtol=1e-5)

# 3. Autograd Gradient Parity
x_param = test_in.clone().detach().requires_grad_(True)
loss = 0.5 * torch.sum((fused_model(x_param) - weights["target"]) ** 2)
loss.backward()
torch.testing.assert_close(fused_model.weight.grad, weights["grad_weight"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(fused_model.bias.grad, weights["grad_bias"], atol=1e-5, rtol=1e-5)
torch.testing.assert_close(x_param.grad, weights["grad_input"], atol=1e-5, rtol=1e-5)
print("All mathematical invariant checks (Inversion + Weight Folding + Gradients) passed.")


In [ ]:
# Micro-benchmark: Sequential (Scaler + Linear) vs Fused Tabular GEMM
def run_sequential():
    z_seq = scaler.transform(test_in)
    return z_seq @ weights["w_head"] + weights["b_head"]

def run_fused():
    return fused_model(test_in)

track_fns = {"Sequential Scaler + Linear": run_sequential, "Fused Tabular GEMM": run_fused}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=test_in.shape[0])
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Data Leakage via Premature Scaling
# 1. Flawed Pattern: Fitting scaler on merged train+test features (Target/Data Leakage)
x_train = test_in[:4]
x_test = test_in[4:]
leaked_mean = torch.mean(test_in, dim=0)  # Contaminated with test set statistics!
isolated_mean = torch.mean(x_train, dim=0)  # Pure train distribution

# Active assertion proving distribution contamination
leak_error = float(torch.max(torch.abs(leaked_mean - isolated_mean)).item())
assert leak_error > 1e-4, "Contamination must alter parameter values"
print(f"[FAILURE DEMONSTRATED] Leaked scaler shifted training mean by {leak_error:.4f}!")

# 2. Correct Pattern: Strict fit-on-train, transform-on-test
clean_scaler = StandardScaler(dim=dim)
clean_scaler.mean.copy_(isolated_mean)
clean_scaler.std.copy_(torch.std(x_train, dim=0, unbiased=False) + 1e-6)
assert torch.equal(clean_scaler.mean, isolated_mean)
print("[CORRECT] Zero-leakage pipeline verified: Scaler fit strictly on training partition.")

# 3. Compilation Verification
ok_exp, msg_exp = core.export.verify_torch_export(fused_model, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(fused_model, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Fused model compiled with torch.compile(fullgraph=True) and exported.")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Tabular Preprocessing & Weight Folding Topology
<svg width="650" height="170" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-t" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">TABULAR PREPROCESSING &amp; SERVING GRAPH FUSION</text>
  <rect x="30" y="50" width="130" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="95" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Raw Tabular X</text>
  <line x1="160" y1="67" x2="220" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-t)"/>
  <rect x="220" y="50" width="170" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="305" y="72" fill="#10b981" font-size="11" text-anchor="middle">Standardize: (X-mu)/sigma</text>
  <line x1="390" y1="67" x2="450" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-t)"/>
  <rect x="450" y="50" width="150" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="525" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Prediction Head</text>
  <rect x="30" y="110" width="570" height="35" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="315" y="132" fill="#f59e0b" font-size="11" text-anchor="middle">Serving Graph Fusion: Y = X @ W_fused + b_fused (1-Pass Zero-DRAM Overhead)</text>
</svg>

### Key Takeaways
1. **Zero-Leakage Discipline:** Preprocessing parameters must be calculated strictly on training partitions to avoid metric contamination.
2. **Serving Graph Fusion:** Folding affine normalization into downstream weights eliminates runtime normalization overhead and memory roundtrips.
3. **Invertible Reconstruction:** `StandardScaler` and `RobustScaler` must support exact numerical inversion for generative and reconstruction tasks.

---
**Curriculum Navigation:**  
&larr; [Previous: 06. Module Internals, Parametrizations & Meta Devices](06_nn_module_internals_parametrizations_and_meta_device.ipynb) | [Master Index: README.md](../../README.md) | Next: 08. Reproducible Preprocessing Pipelines & Determinism (Upcoming) &rarr;
